# Hafta 14 · Kuantum Makine Öğrenmesi III: Varyasyonel Sınıflandırıcılar, Kuantum Sinir Ağları ve Hibrit Modeller
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab (CPU yeterli)

Bu hafta **eğitilebilir kuantum devrelerini** bir sınıflandırıcı gibi kullanıyoruz. PennyLane ile devreyi yazacak, PyTorch ile eğitecek, klasik modellerle dürüstçe karşılaştıracak ve derin devrelerin neden eğitilemez hâle geldiğini (çorak düzlük, *barren plateau*) sayısal olarak göreceğiz.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum, veri setleri, yardımcılar | 4 dk |
| A | PennyLane temelleri: device, QNode, şablonlar, çizim, gradyan | 8 dk |
| B | VQC'yi sıfırdan yazmak ve PyTorch ile eğitmek (moons) | 10 dk |
| C | Katman sayısı deneyi (iris2, 4 kübit) | 5 dk |
| D | Hibrit model: `qml.qnn.TorchLayer` + `nn.Sequential` | 6 dk |
| E | Veri yeniden yükleme (data re-uploading): tek kübitle doğrusal olmayan sınır | 6 dk |
| F | Klasik modellerle karşılaştırma (doğruluk, parametre, süre) | 4 dk |
| G | Ağırlık başlatmanın etkisi | 3 dk |
| H | Çorak düzlük (barren plateau) deneyi | 4 dk |
| I | Alıştırmalar (8 adet, `assert` ile kendini kontrol) | ödev |

> Bit sırası kuralı derste olduğu gibi Qiskit sırasıdır (q₀ en sağda). PennyLane çizimlerinde ise tel 0 en üsttedir; bu yalnızca bir gösterim farkıdır.

## 0 · Kurulum
PyTorch Colab'da hazır gelir; **kurmayın**. Sadece PennyLane ve devre çizimi için Qiskit kuruyoruz.

In [ ]:
!pip install -q pennylane qiskit pylatexenc

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch, torch.nn as nn
import pennylane as qml
from pennylane import numpy as pnp          # PennyLane'in otomatik türev destekli NumPy'si
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from sklearn.datasets import make_moons, make_circles, load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")

torch.set_default_dtype(torch.float64)      # kuantum simülasyonu çift duyarlıkla daha kararlı
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
CM = LinearSegmentedColormap.from_list("w14", ["#F6D9BE", "#FFFFFF", "#C9DAF0"])
print("PennyLane", qml.__version__, "| torch", torch.__version__)

### Veri setleri
Aşağıdaki fonksiyonlar dersin ortak veri seti kodudur (her hafta aynı). Tüm özellikler **[0, π]** aralığına ölçeklenmiştir, yani doğrudan açı olarak kodlanabilir.
Bu veri setlerinin CSV'leri ayrıca verilmiştir: `moons.csv`, `circles.csv`, `iris_2sinif.csv`.

In [ ]:
SEED = 42

def ds_moons(n=200, noise=0.15):
    X, y = make_moons(n_samples=n, noise=noise, random_state=SEED)
    X = MinMaxScaler((0, np.pi)).fit_transform(X)            # açı kodlaması için [0, π]
    return pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y})

def ds_circles(n=200, noise=0.08, factor=0.45):
    X, y = make_circles(n_samples=n, noise=noise, factor=factor, random_state=SEED)
    X = MinMaxScaler((0, np.pi)).fit_transform(X)
    return pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y})

def ds_iris2():
    """Iris: versicolor (0) ve virginica (1) — doğrusal olarak tam ayrılamayan iki sınıf; 4 özellik."""
    d = load_iris()
    m = d.target > 0
    X = MinMaxScaler((0, np.pi)).fit_transform(d.data[m])
    y = d.target[m] - 1
    cols = ["sepal_len", "sepal_wid", "petal_len", "petal_wid"]
    df = pd.DataFrame(X, columns=cols); df["y"] = y
    return df

def split(df):
    """%70 eğitim / %30 test, sınıf oranları korunur (stratify)."""
    X = df.drop(columns="y").values; y = df.y.values
    return train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

Xm_tr, Xm_te, ym_tr, ym_te = split(ds_moons())
Xc_tr, Xc_te, yc_tr, yc_te = split(ds_circles())
Xi_tr, Xi_te, yi_tr, yi_te = split(ds_iris2())
print("moons  :", Xm_tr.shape, Xm_te.shape)
print("circles:", Xc_tr.shape, Xc_te.shape)
print("iris2  :", Xi_tr.shape, Xi_te.shape)

fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, df, t, a, b in [(axs[0], ds_moons(), "moons", "x1", "x2"), (axs[1], ds_circles(), "circles", "x1", "x2"), (axs[2], ds_iris2(), "iris2", "petal_len", "petal_wid")]:
    ax.scatter(df[a], df[b], c=df.y.map({0: ORANGE, 1: BLUE}), s=12); ax.set_title(t, color=NAVY); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

In [ ]:
# Karar sınırı çizimi için yardımcı (2 özellikli veri setleri)
GX = np.linspace(0, np.pi, 50)
GRID = np.array([[a, b] for b in GX for a in GX])

def plot_boundary(prob_fn, X, y, title, ax=None):
    """prob_fn: (N,2) numpy dizisi alır, P(y=1) döndürür."""
    show = ax is None
    if show: fig, ax = plt.subplots(figsize=(4, 4))
    Z = prob_fn(GRID).reshape(50, 50)
    ax.contourf(GX, GX, Z, levels=np.linspace(0, 1, 11), cmap=CM)
    ax.contour(GX, GX, Z, levels=[0.5], colors=NAVY, linewidths=1.8)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 1, BLUE, ORANGE), s=12, edgecolors="white", linewidths=0.3)
    ax.set_title(title, color=NAVY, fontsize=11); ax.set_aspect("equal")
    if show: plt.show()

def accuracy(p, y):
    return float(((np.asarray(p) > 0.5) == (np.asarray(y) == 1)).mean())
print("hazır")

---
## A · PennyLane temelleri
PennyLane'de bir kuantum devresi **sıradan bir Python fonksiyonudur**. `@qml.qnode(dev)` dekoratörü bu fonksiyonu bir cihaza (simülatör veya donanım) bağlar ve **türevi alınabilir** hâle getirir.

| Kavram | PennyLane | Qiskit karşılığı |
|---|---|---|
| Simülatör | `qml.device("default.qubit", wires=2)` | `StatevectorEstimator()` / `AerSimulator()` |
| Devre | `@qml.qnode(dev)` ile süslenmiş fonksiyon | `QuantumCircuit(2)` |
| Kapı | `qml.RY(theta, wires=0)` | `qc.ry(theta, 0)` |
| CNOT | `qml.CNOT(wires=[0, 1])` | `qc.cx(0, 1)` |
| Çıktı | `return qml.expval(qml.PauliZ(0))` | `SparsePauliOp("ZI")` gözlemlenebiliri + Estimator |
| Gradyan | `qml.grad(f)` veya `torch.autograd` | `ParamShiftEstimatorGradient` (qiskit-machine-learning) |
| Çizim | `qml.draw(f)(...)`, `qml.draw_mpl(f)(...)` | `qc.draw("mpl")` |

In [ ]:
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def tek_kubit(x, theta):
    qml.RY(x, wires=0)          # kodlama: veriyi açıya yaz
    qml.RY(theta, wires=0)      # eğitilebilir kapı
    return qml.expval(qml.PauliZ(0))

print("⟨Z⟩ =", tek_kubit(0.8, 1.2), "   elle: cos(0.8 + 1.2) =", np.cos(2.0))
print(qml.draw(tek_kubit)(0.8, 1.2))

### Hazır şablonlar (templates)
- `qml.AngleEmbedding(x, wires, rotation="Y")`: her özelliği bir kübite RY açısı olarak yazar (12. haftadaki açı kodlaması).
- `qml.BasicEntanglerLayers(w, wires, rotation=qml.RY)`: her katmanda her kübite bir döndürme + CNOT halkası. Ağırlık şekli `(L, n)`.
- `qml.StronglyEntanglingLayers(w, wires)`: her kübite genel `Rot(φ, θ, ω)` + farklı mesafeli CNOT'lar. Ağırlık şekli `(L, n, 3)`.

In [ ]:
n = 3
dev3 = qml.device("default.qubit", wires=n)

@qml.qnode(dev3)
def sablon_devre(x, w):
    qml.AngleEmbedding(x, wires=range(n), rotation="Y")
    qml.StronglyEntanglingLayers(w, wires=range(n))
    return qml.expval(qml.PauliZ(0))

print("BasicEntanglerLayers ağırlık şekli (L=2):", (2, n))
print("StronglyEntanglingLayers ağırlık şekli (L=2):", qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=n))
x = np.array([0.1, 0.2, 0.3]); w = np.full((2, n, 3), 0.5)
print("çıktı:", sablon_devre(x, w))
fig, ax = qml.draw_mpl(sablon_devre, style="black_white", level="device")(x, w)
plt.show()

### Aynı devre Qiskit'te
PennyLane'deki `AngleEmbedding` + tek katman `RY·RZ` + CNOT devresinin Qiskit karşılığı:

In [ ]:
xp, tp = ParameterVector("x", 2), ParameterVector("θ", 4)
qc = QuantumCircuit(2)
qc.ry(xp[0], 0); qc.ry(xp[1], 1); qc.barrier()
qc.ry(tp[0], 0); qc.rz(tp[1], 0); qc.ry(tp[2], 1); qc.rz(tp[3], 1); qc.cx(0, 1)
qc.draw("mpl")

### Gradyan: parameter-shift ve otomatik türev (backprop)
11\. haftada gördüğümüz **parameter-shift** kuralı gerçek donanımda da çalışır: bir parametreyi ±π/2 kaydırıp devreyi iki kez çalıştırırız.
Simülatörde ise PennyLane durum vektörü üzerindeki tüm işlemleri PyTorch/Autograd gibi **otomatik türev** ile izleyebilir (`diff_method="backprop"`): çok daha hızlıdır ama sadece simülatörde mümkündür.

In [ ]:
theta = pnp.array(1.2, requires_grad=True)
qn_ps = qml.QNode(tek_kubit.func, dev, diff_method="parameter-shift")
qn_bp = qml.QNode(tek_kubit.func, dev, diff_method="backprop")

g_ps = qml.grad(qn_ps, argnums=1)(0.8, theta)
g_bp = qml.grad(qn_bp, argnums=1)(0.8, theta)
g_el = (tek_kubit(0.8, 1.2 + np.pi/2) - tek_kubit(0.8, 1.2 - np.pi/2)) / 2
print(f"parameter-shift (PennyLane): {g_ps:.6f}")
print(f"backprop (PennyLane)       : {g_bp:.6f}")
print(f"parameter-shift (elle)     : {g_el:.6f}")
print(f"analitik  −sin(2.0)        : {-np.sin(2.0):.6f}")

---
## B · VQC'yi sıfırdan yazmak (moons, 2 kübit)
**Anatomi:** `kodlama S(x)` → `ansatz U(θ)` (L katman: her kübite RY·RZ + CNOT) → `⟨Z⟩` ölçümü → olasılık `p = (1 − ⟨Z⟩)/2`.

`p`, son kübitin **1 okunma olasılığıdır** (⟨Z⟩ = P(0) − P(1) olduğundan). Kayıp: ikili çapraz entropi (BCE). Optimizer: Adam. Mini-batch: 16.

`interface="torch"` sayesinde QNode PyTorch tensörü alıp döndürür; `loss.backward()` gradyanı devrenin içinden geçirerek hesaplar. Üstelik girdiye `(batch, özellik)` şekilli bir tensör verirsek PennyLane **tüm batch'i tek çağrıda** (yayınlama, *broadcasting*) hesaplar.

In [ ]:
def make_vqc(n, L):
    dev = qml.device("default.qubit", wires=n)
    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circ(x, w):
        qml.AngleEmbedding(x, wires=range(n), rotation="Y")       # kodlama katmanı
        for l in range(L):                                         # eğitilebilir ansatz
            for i in range(n):
                qml.RY(w[l, i, 0], wires=i)
                qml.RZ(w[l, i, 1], wires=i)
            for i in range(n if n > 2 else 1):                     # CNOT halkası (n=2'de tek CNOT)
                qml.CNOT(wires=[i, (i + 1) % n])
        return qml.expval(qml.PauliZ(n - 1))                       # son kübitin ⟨Z⟩'si
    return circ

vqc = make_vqc(2, 2)
w0 = torch.zeros(2, 2, 2)
print(qml.draw(vqc)(torch.tensor([0.3, 1.1]), w0))
print("batch çıktısı şekli:", vqc(torch.tensor(Xm_tr[:5]), w0).shape)

In [ ]:
def train_vqc(circ, wshape, Xtr, ytr, Xte, yte, epochs=25, lr=0.1, bs=16, seed=0, init="uniform", verbose=True):
    g = torch.Generator().manual_seed(seed)
    if init == "uniform": w = torch.rand(wshape, generator=g) * 2 * np.pi
    elif init == "small": w = torch.randn(wshape, generator=g) * 0.1
    else: w = torch.zeros(wshape)
    w.requires_grad_(True)
    opt = torch.optim.Adam([w], lr=lr)
    Xtr_t, Xte_t = torch.tensor(Xtr), torch.tensor(Xte)
    ytr_t, yte_t = torch.tensor(ytr, dtype=torch.float64), torch.tensor(yte, dtype=torch.float64)
    prob = lambda X: torch.clamp((1 - circ(X, w)) / 2, 1e-6, 1 - 1e-6)   # ⟨Z⟩ → p
    bce = nn.functional.binary_cross_entropy
    hist = []; t0 = time.time()
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr_t), generator=g)
        for i in range(0, len(perm), bs):                      # mini-batch döngüsü
            idx = perm[i:i + bs]
            opt.zero_grad()
            loss = bce(prob(Xtr_t[idx]), ytr_t[idx])
            loss.backward()                                    # gradyan (backprop)
            opt.step()                                         # Adam adımı
        with torch.no_grad():
            ptr, pte = prob(Xtr_t), prob(Xte_t)
            hist.append([bce(ptr, ytr_t).item(), bce(pte, yte_t).item(), accuracy(ptr, ytr), accuracy(pte, yte)])
        if verbose and (ep % 5 == 4 or ep == 0):
            print(f"epoch {ep+1:2d}  kayıp {hist[-1][0]:.3f}/{hist[-1][1]:.3f}  doğruluk {hist[-1][2]:.3f}/{hist[-1][3]:.3f}")
    predict = lambda X: prob(torch.tensor(X)).detach().numpy()
    return np.array(hist), time.time() - t0, w.detach(), predict

hist, sure, w_m, pred_m = train_vqc(make_vqc(2, 2), (2, 2, 2), Xm_tr, ym_tr, Xm_te, ym_te)
print(f"süre: {sure:.1f} s   test doğruluğu: {hist[-1, 3]:.3f}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
ep = np.arange(1, len(hist) + 1)
axs[0].plot(ep, hist[:, 0], color=BLUE, label="eğitim"); axs[0].plot(ep, hist[:, 1], color=ORANGE, ls="--", label="test")
axs[0].set_title("BCE kaybı", color=NAVY); axs[0].legend(frameon=False)
axs[1].plot(ep, hist[:, 2], color=BLUE, label="eğitim"); axs[1].plot(ep, hist[:, 3], color=ORANGE, ls="--", label="test")
axs[1].set_title("doğruluk", color=NAVY); axs[1].legend(frameon=False)
for a in axs[:2]: a.set_xlabel("epoch"); a.grid(alpha=0.25)
plot_boundary(pred_m, Xm_te, ym_te, "VQC L=2 karar sınırı (test noktaları)", ax=axs[2])
plt.tight_layout(); plt.show()

---
## C · Katman sayısı deneyi (iris2, 4 kübit)
iris2 dört özelliklidir → **4 kübit**. Parametre sayısı: `2 · n · L` (her kübit ve katmanda bir RY ve bir RZ).

In [ ]:
sonuc_C = []
for L in [1, 2, 3]:
    h, t, _, _ = train_vqc(make_vqc(4, L), (L, 4, 2), Xi_tr, yi_tr, Xi_te, yi_te, epochs=20, verbose=False)
    sonuc_C.append({"L": L, "parametre": 2 * 4 * L, "eğitim doğ.": round(h[-1, 2], 3), "test doğ.": round(h[-1, 3], 3), "süre (s)": round(t, 1)})
pd.DataFrame(sonuc_C)

---
## D · Hibrit model: `qml.qnn.TorchLayer`
`TorchLayer`, bir QNode'u **sıradan bir `nn.Module`** hâline getirir. Böylece kuantum katmanını klasik katmanların arasına koyup tüm modeli tek bir PyTorch döngüsüyle eğitebiliriz.

Kurallar (PennyLane 0.45):
1. QNode'un veri argümanının adı **`inputs`** olmalıdır.
2. Eğitilebilir argümanların şekilleri `weight_shapes` sözlüğüyle verilir: `{"weights": (L, n, 3)}`.
3. QNode birden çok ⟨Z⟩ döndürürse katmanın çıktı boyutu kübit sayısı olur.

In [ ]:
n_q = 2
dev_h = qml.device("default.qubit", wires=n_q)

@qml.qnode(dev_h, interface="torch", diff_method="backprop")
def qkatman(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_q), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_q))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_q)]

def make_hybrid(seed=0):
    torch.manual_seed(seed)
    ql = qml.qnn.TorchLayer(qkatman, {"weights": (2, n_q, 3)})
    return nn.Sequential(nn.Linear(2, n_q), ql, nn.Linear(n_q, 1))

hybrid = make_hybrid()
print(hybrid)
print("toplam parametre:", sum(p.numel() for p in hybrid.parameters()))

In [ ]:
def train_torch(model, Xtr, ytr, Xte, yte, epochs=20, lr=0.05, bs=16, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss()                 # sigmoid + BCE birlikte (sayısal olarak kararlı)
    Xt, yt = torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.float64).unsqueeze(1)
    Xv, yv = torch.tensor(Xte), torch.tensor(yte, dtype=torch.float64).unsqueeze(1)
    hist = []; t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]
            opt.zero_grad(); loss = lossf(model(Xt[idx]), yt[idx]); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            a, b = model(Xt), model(Xv)
            hist.append([lossf(a, yt).item(), lossf(b, yv).item(), accuracy(torch.sigmoid(a).ravel(), ytr), accuracy(torch.sigmoid(b).ravel(), yte)])
    return np.array(hist), time.time() - t0

h_hyb, t_hyb = train_torch(hybrid, Xm_tr, ym_tr, Xm_te, ym_te)
print(f"hibrit model: test doğruluğu {h_hyb[-1, 3]:.3f}, süre {t_hyb:.1f} s")
pred_h = lambda X: torch.sigmoid(hybrid(torch.tensor(X))).detach().numpy().ravel()
plot_boundary(pred_h, Xm_te, ym_te, "Hibrit model karar sınırı")

---
## E · Veri yeniden yükleme (data re-uploading)
Tek kübit + tek kodlama yalnızca `cos(x + θ)` gibi **basit** fonksiyonlar üretebilir. Çözüm: veriyi devreye **birden çok kez** yüklemek, arada eğitilebilir döndürmeler koymak (Pérez-Salinas ve diğ., 2020). Her katman:

`RY(w₀·x₁ + w₁) → RZ(w₂·x₂ + w₃)`

Böylece çıktı, katman sayısıyla zenginleşen bir **trigonometrik polinom** olur (bir Fourier serisi gibi). Özellikleri önce π/2 çevresine kaydırıyoruz (`x − π/2`), böylece ölçek ağırlıkları (w₀, w₂) simetrik çalışır.

In [ ]:
def make_reup(L):
    dev = qml.device("default.qubit", wires=1)
    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circ(x, w):
        xc = x - np.pi / 2
        for l in range(L):
            qml.RY(w[l, 0] * xc[..., 0] + w[l, 1], wires=0)   # veri yeniden yükleniyor
            qml.RZ(w[l, 2] * xc[..., 1] + w[l, 3], wires=0)
        qml.RY(w[L, 0], wires=0)                               # son ayar döndürmesi
        return qml.expval(qml.PauliZ(0))
    return circ

fig, axs = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, L in zip(axs, [1, 2, 3]):
    h, t, _, pr = train_vqc(make_reup(L), (L + 1, 4), Xc_tr, yc_tr, Xc_te, yc_te, epochs=30, seed=1, verbose=False)
    plot_boundary(pr, Xc_te, yc_te, f"L = {L}, {4*L+1} param., test = {h[-1,3]:.2f}", ax=ax)
plt.suptitle("circles: TEK kübit, veri L kez yükleniyor", color=NAVY); plt.tight_layout(); plt.show()

---
## F · Klasik modellerle karşılaştırma
Dürüst bir karşılaştırma için **aynı eğitim/test bölmesini** kullanıyoruz ve doğruluk, parametre sayısı ve eğitim süresini birlikte raporluyoruz.

In [ ]:
satirlar = []
t0 = time.time(); lr_m = LogisticRegression().fit(Xm_tr, ym_tr); t_lr = time.time() - t0
t0 = time.time(); mlp = MLPClassifier((16,), max_iter=3000, random_state=0).fit(Xm_tr, ym_tr); t_mlp = time.time() - t0
satirlar.append(["Lojistik regresyon", 3, lr_m.score(Xm_te, ym_te), t_lr])
satirlar.append(["MLP (16 gizli nöron)", 2*16 + 16 + 16 + 1, mlp.score(Xm_te, ym_te), t_mlp])
satirlar.append(["VQC (2 kübit, L=2)", 8, hist[-1, 3], sure])
satirlar.append(["Hibrit (Linear→Kuantum→Linear)", 21, h_hyb[-1, 3], t_hyb])
tablo = pd.DataFrame(satirlar, columns=["model", "parametre", "test doğruluğu", "eğitim süresi (s)"]).round(3)
tablo

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
plot_boundary(lambda X: lr_m.predict_proba(X)[:, 1], Xm_te, ym_te, "Lojistik regresyon", ax=axs[0])
plot_boundary(lambda X: mlp.predict_proba(X)[:, 1], Xm_te, ym_te, "MLP (16 nöron)", ax=axs[1])
plt.tight_layout(); plt.show()

**Yorum:** Bu küçük ve düşük boyutlu veri setlerinde kuantum modeller klasik modelleri **geçmiyor**; ancak çok az parametreyle benzer doğruluğa ulaşıyor. Simülatörde eğitim süresi de klasik modellerden belirgin şekilde uzundur. "Kuantum avantajı" sorusunu 15. haftada ayrıntılı tartışacağız.

---
## G · Ağırlık başlatmanın etkisi
Aynı modeli üç farklı başlatmayla eğitelim: düzgün `U(0, 2π)`, küçük `N(0, 0.1²)` ve sıfır. Derin (L = 6) bir 4 kübitlik devre kullanıyoruz.

In [ ]:
plt.figure(figsize=(7, 3.6))
for init, col in [("uniform", BLUE), ("small", NAVY), ("zeros", ORANGE)]:
    h, _, _, _ = train_vqc(make_vqc(4, 6), (6, 4, 2), Xi_tr, yi_tr, Xi_te, yi_te, epochs=10, lr=0.05, init=init, verbose=False)
    plt.plot(np.arange(1, 11), h[:, 0], color=col, lw=2, label=f"{init}: son kayıp {h[-1,0]:.3f}")
plt.xlabel("epoch"); plt.ylabel("eğitim kaybı"); plt.legend(frameon=False); plt.grid(alpha=0.25)
plt.title("Başlatma stratejileri", color=NAVY); plt.show()

Sıfır başlatmada devre başlangıçta yalnızca kodlama + CNOT'lardan oluşur ve ilk adımlar yavaş olabilir; küçük rastgele başlatma hem simetriyi kırar hem de devreyi "kimliğe yakın" (sığ davranan) bir noktadan başlatır. Bu, çorak düzlükten kaçınmanın da bir yoludur (Bölüm H).

---
## H · Çorak düzlük (barren plateau)
**Soru:** Rastgele başlatılmış derin bir devrede tek bir parametrenin gradyanı ne kadar büyük?

Deney: her n için 200 rastgele parametre vektörü üretiyoruz, ∂C/∂θ₁'i parameter-shift ile hesaplıyoruz ve **varyansına** bakıyoruz. Ortalama zaten ≈ 0; varyans küçükse gradyan neredeyse her yerde sıfırdır → optimizer yön bulamaz.

Hız için PennyLane'in **yayınlama (broadcasting)** özelliğini kullanıyoruz: parametre dizisinin son boyutu 200 olunca devre 200 kez tek çağrıda simüle edilir.

In [ ]:
def bp_circuit(n, L, glob=True):
    dev = qml.device("default.qubit", wires=n)
    @qml.qnode(dev)
    def f(w):
        for i in range(n): qml.RY(np.pi / 4, wires=i)
        for l in range(L):
            for i in range(n):
                qml.RY(w[l, i, 0], wires=i); qml.RZ(w[l, i, 1], wires=i)
            for i in range(0, n - 1, 2): qml.CZ(wires=[i, i + 1])   # "tuğla" dolanıklık
            for i in range(1, n - 1, 2): qml.CZ(wires=[i, i + 1])
        if glob:
            return qml.expval(qml.prod(*[qml.PauliZ(i) for i in range(n)]))   # global maliyet
        return qml.expval(qml.PauliZ(0))                                      # yerel maliyet
    return f

def grad_variance(n, L, glob=True, B=200, seed=0, init="uniform"):
    rng = np.random.default_rng(seed)
    f = bp_circuit(n, L, glob)
    W = rng.uniform(0, 2*np.pi, (L, n, 2, B)) if init == "uniform" else rng.normal(0, 0.1, (L, n, 2, B))
    Wp, Wm = W.copy(), W.copy()
    Wp[0, 0, 0] += np.pi / 2; Wm[0, 0, 0] -= np.pi / 2
    grads = (f(Wp) - f(Wm)) / 2                    # parameter-shift, 200 örnek birden
    return float(np.var(grads))

t0 = time.time()
ns = [2, 3, 4, 5, 6, 7, 8]
var_g = [grad_variance(n, 10, True) for n in ns]
var_l = [grad_variance(n, 10, False) for n in ns]
print(f"süre: {time.time()-t0:.1f} s")
pd.DataFrame({"n": ns, "Var (global)": var_g, "Var (yerel)": var_l})

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy(ns, var_g, "o-", color=NAVY, lw=2, label="global maliyet")
plt.semilogy(ns, var_l, "s-", color=BLUE, lw=2, label="yerel maliyet")
plt.semilogy(ns, 0.5 * 2.0 ** -np.array(ns), ":", color=ORANGE, lw=2, label="∝ 2⁻ⁿ")
plt.xlabel("kübit sayısı n (L = 10)"); plt.ylabel("Var[∂C/∂θ₁] (log)"); plt.grid(alpha=0.25, which="both"); plt.legend(frameon=False)
plt.title("Çorak düzlük: log ölçekte düz bir çizgi = üstel azalma", color=NAVY); plt.show()

eğim = np.polyfit(ns, np.log2(var_g), 1)[0]
print(f"global maliyet: her ek kübitte varyans yaklaşık 2^{eğim:.2f} = {2**eğim:.2f} katına iniyor")

**Çözüm önerileri:** sığ devre (az katman), **yerel maliyet** (tek kübitin ⟨Z⟩'si), **akıllı başlatma** (küçük açılar), **katman katman eğitim** (önce 1 katman eğit, sonra yenisini ekle). Bunların etkisini Alıştırma 7–8'de ölçeceksiniz.

---
## I · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur. Tohumlar sabittir, sonuçlar tekrarlanabilir.

### Alıştırma 1 · ⟨Z⟩ → olasılık → BCE
`p_from_expval(z)` fonksiyonu p = (1 − z)/2 döndürsün; `bce(p, y)` tek örnek için −[y·log p + (1 − y)·log(1 − p)] hesaplasın. 1 kübitlik devre `RY(0.8)·RY(1.2)|0⟩` için gerçek sınıf y = 1 iken kaybı bulun.

In [ ]:
def p_from_expval(z):
    # TODO
    pass

def bce(p, y):
    # TODO
    pass

z = float(tek_kubit(0.8, 1.2))
p = p_from_expval(z)
L = bce(p, 1)
print(f"⟨Z⟩ = {z:.4f}, p = {p:.4f}, BCE = {L:.4f}")
assert np.isclose(p, 0.708073, atol=1e-5)
assert np.isclose(L, 0.345207, atol=1e-5)
assert np.isclose(bce(0.9, 0), -np.log(0.1))
print("Alıştırma 1 ✓")

### Alıştırma 2 · Parametre sayısı
`n_params(n, L, kind)` fonksiyonunu yazın: `kind="basic"` → BasicEntanglerLayers, `"ryrz"` → bizim RY+RZ ansatz'ımız, `"strong"` → StronglyEntanglingLayers. Sonucu PennyLane'in `shape` fonksiyonu ve hibrit modelin gerçek parametre sayısıyla karşılaştırın.

In [ ]:
def n_params(n, L, kind):
    # TODO
    pass

assert n_params(4, 3, "basic") == int(np.prod(qml.BasicEntanglerLayers.shape(n_layers=3, n_wires=4)))
assert n_params(4, 3, "strong") == int(np.prod(qml.StronglyEntanglingLayers.shape(n_layers=3, n_wires=4)))
assert n_params(2, 2, "ryrz") == 8
# hibrit: Linear(2,2) + kuantum (L=2, n=2, strong) + Linear(2,1)
assert (2*2 + 2) + n_params(2, 2, "strong") + (2 + 1) == sum(p.numel() for p in make_hybrid().parameters())
print("Alıştırma 2 ✓")

### Alıştırma 3 · Parameter-shift gradyanını elle yazmak
`ps_grad(f, w)` fonksiyonu, `f(w)` skaler bir QNode çıktısı olmak üzere **tüm** parametrelerin gradyanını parameter-shift kuralıyla (her parametre için 2 çalıştırma) hesaplasın. `pnp` otomatik türeviyle karşılaştırın.

In [ ]:
dev2 = qml.device("default.qubit", wires=2)
@qml.qnode(dev2)
def f2(w):
    qml.RY(0.4, wires=0); qml.RY(1.0, wires=1)
    qml.RY(w[0], wires=0); qml.RZ(w[1], wires=0); qml.RY(w[2], wires=1); qml.RZ(w[3], wires=1)
    qml.CNOT(wires=[0, 1])
    return qml.expval(qml.PauliZ(1))

def ps_grad(f, w):
    # TODO
    pass

w = pnp.array([0.3, -0.5, 0.9, 0.2], requires_grad=True)
g_ps = ps_grad(f2, np.array(w))
g_ad = qml.grad(f2)(w)
print("parameter-shift:", g_ps, "\notomatik türev :", g_ad)
assert np.allclose(g_ps, g_ad, atol=1e-8)
print("Alıştırma 3 ✓")

### Alıştırma 4 · StronglyEntanglingLayers ile iris2 sınıflandırıcısı
`make_vqc_strong(n, L)` fonksiyonunu yazın: `AngleEmbedding` (RY) + `StronglyEntanglingLayers` + son kübitin ⟨Z⟩'si. `train_vqc` ile iris2 üzerinde L = 2, 15 epoch eğitin. Test doğruluğu ≥ 0.80 olmalı.

In [ ]:
def make_vqc_strong(n, L):
    dev = qml.device("default.qubit", wires=n)
    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circ(x, w):
        # TODO
        pass
    return circ

h4, t4, _, _ = train_vqc(make_vqc_strong(4, 2), (2, 4, 3), Xi_tr, yi_tr, Xi_te, yi_te, epochs=15, lr=0.1, verbose=False)
print(f"test doğruluğu: {h4[-1, 3]:.3f}  süre: {t4:.1f} s")
assert h4[-1, 3] >= 0.80
print("Alıştırma 4 ✓")

### Alıştırma 5 · Data re-uploading ile moons
Bölüm E'deki `make_reup` modelini **moons** üzerinde L = 3 ile (seed = 0, 30 epoch) eğitin ve karar sınırını çizin. Test doğruluğu ≥ 0.85 olmalı. Parametre sayısını da `reup_params(L)` fonksiyonuyla hesaplayın.

In [ ]:
def reup_params(L):
    # TODO
    pass

# TODO: h5, _, _, pr5 = train_vqc(...)
h5, pr5 = None, None

print(f"parametre: {reup_params(3)}, test doğruluğu: {h5[-1, 3]:.3f}")
plot_boundary(pr5, Xm_te, ym_te, "moons, tek kübit re-uploading L=3")
assert reup_params(3) == 13
assert h5[-1, 3] >= 0.85
print("Alıştırma 5 ✓")

### Alıştırma 6 · Hibrit model, 4 özellik
iris2 için bir hibrit model kurun: `nn.Linear(4, 2)` → 2 kübitlik `TorchLayer` (AngleEmbedding + StronglyEntanglingLayers, L = 2) → `nn.Linear(2, 1)`. `train_torch` ile 20 epoch eğitin (seed = 0). Parametre sayısı 25, test doğruluğu ≥ 0.80 olmalı.

In [ ]:
torch.manual_seed(0)
# TODO: hyb4 = nn.Sequential(...)
hyb4 = None

h6, t6 = train_torch(hyb4, Xi_tr, yi_tr, Xi_te, yi_te, epochs=20, seed=0)
n6 = sum(p.numel() for p in hyb4.parameters())
print(f"parametre: {n6}, test doğruluğu: {h6[-1, 3]:.3f}, süre: {t6:.1f} s")
assert n6 == 25
assert h6[-1, 3] >= 0.80
print("Alıştırma 6 ✓")

### Alıştırma 7 · Yerel maliyet çorak düzlüğü hafifletir
`grad_variance` ile n = 6, L = 4 için global ve yerel maliyetin gradyan varyansını hesaplayın. Yerel maliyetin varyansı en az 5 kat büyük olmalı.

In [ ]:
# TODO
v_glob, v_loc = None, None

print(f"global: {v_glob:.2e}   yerel: {v_loc:.2e}   oran: {v_loc / v_glob:.1f}")
assert v_loc > 5 * v_glob
print("Alıştırma 7 ✓")

### Alıştırma 8 · Akıllı başlatma
n = 8, L = 20, **global** maliyet için düzgün `U(0, 2π)` başlatma ile küçük `N(0, 0.1²)` başlatmanın gradyan varyanslarını karşılaştırın (`init="small"`). Ayrıca düzgün başlatmada n = 2'den n = 8'e varyansın en az 20 kat düştüğünü gösterin.

In [ ]:
# TODO
v_uni8, v_small8, v_uni2 = None, None, None

print(f"n=8 düzgün: {v_uni8:.2e}   n=8 küçük: {v_small8:.2e}   n=2 düzgün: {v_uni2:.2e}")
assert v_uni2 > 20 * v_uni8
assert v_small8 > v_uni8
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- **VQC = kodlama + eğitilebilir ansatz + ⟨Z⟩ ölçümü**; olasılık p = (1 − ⟨Z⟩)/2, kayıp BCE, optimizer Adam
- Parametre sayısı: RY+RZ ansatz'ta 2·n·L, StronglyEntanglingLayers'ta 3·n·L
- Gradyan: donanımda **parameter-shift** (2 çalıştırma/parametre), simülatörde **backprop** (çok daha hızlı)
- `qml.qnn.TorchLayer` ile kuantum devresi sıradan bir PyTorch katmanı olur (`inputs` argüman adı zorunlu)
- **Data re-uploading**: tek kübitle bile doğrusal olmayan karar sınırları
- Küçük veri setlerinde klasik modeller en az kuantum modeller kadar iyi ve çok daha hızlı
- **Çorak düzlük**: derin, rastgele başlatılmış devrelerde gradyan varyansı kübit sayısıyla üstel azalır → sığ devre, yerel maliyet, küçük başlatma, katman katman eğitim

**Gelecek hafta (Hafta 15):** Uçtan uca bir QML projesi (veri hazırlama → kodlama → model → değerlendirme), klasik ve kuantum modellerin adil karşılaştırması ve "kuantum avantajı gerçekte var mı?" tartışması; dönem projesi önerileri.